# Восстановление пунктуации (мультимодальное, late fusion) — корпус M-AILABS

Модель преобразует **текст без пунктуации → текст с пунктуацией**, опираясь *одновременно* на текст и на акустические признаки из звука (паузы, длительности, темп, **F0**, энергия). Восстанавливаются **запятые, точки, многоточия, вопросительные и восклицательные знаки**, а также **абзацы (красные строки)** и **капитализация**.

## Архитектуры
BiLSTM (baseline) · Transformer с нуля (baseline-трансформер) · RuBERT-base и rubert-tiny2 (предобученные). Все двухпоточные: текст + акустика.

## 0. Зависимости

In [ ]:
# --- УСТАНОВКА (раскомментируйте при первом запуске) ---
# import sys
# Зависимости проекта:
# !{sys.executable} -m pip install -r requirements.txt
#
# Forced aligner (акустика: паузы/F0). ВАЖНО: ставьте ТЕМ ЖЕ python, что у ядра,
# иначе ядро его не увидит. ffmpeg обязателен.
# !{sys.executable} -m pip install git+https://github.com/MahmoudAshraf97/ctc-forced-aligner.git
# Windows: ffmpeg через conda -> conda install -c conda-forge ffmpeg
# Linux:   sudo apt install ffmpeg

import os
os.environ.setdefault("DATASETS_AUDIO_BACKEND", "soundfile")

## 1. Импорт модулей

In [2]:
import numpy as np
import torch
from functools import partial
from torch.utils.data import DataLoader

from modules.models.__init__ import *
from modules.models.heads import *
from modules.models.lstm_model import *
from modules.models.pretrained_model import *
from modules.models.transformer_model import *
from modules.__init__ import *
from modules.config import *
from modules.data import *
from modules.dataset import *
from modules.evaluate import *
from modules.inference import *
from modules.tokenizer import *
from modules.train import *


cfg = get_config()
set_seed(cfg.train.seed)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
cfg.train.device = DEVICE
print("device:", DEVICE)
print("пунктуация:", PUNCT_LABELS, "| абзац:", PARA_LABELS, "| капитализация:", CAP_LABELS)
print("акустика:", ACOUSTIC_FEATURES)
print("loss:", cfg.train.loss_type, "| автовеса классов:", cfg.train.auto_class_weights)

device: cpu
пунктуация: ['O', 'COMMA', 'PERIOD', 'QUESTION', 'EXCLAM', 'ELLIPSIS'] | абзац: ['NO_PARA', 'PARA'] | капитализация: ['LOWER', 'CAP', 'UPPER']
акустика: ['pause_before', 'pause_after', 'word_duration', 'speech_rate', 'f0_end_median', 'f0_end_slope', 'energy_end']
loss: focal | автовеса классов: True


## 1b. Самодиагностика окружения

In [3]:
diagnose()

modules.__version__ = 2.3-mailabs
python (ядро)       = c:\Users\Roman\Documents\Projects\STT_russian_lang\venv\Scripts\python.exe
mailabs_repos       = ['psiyou/m-ailabs-ru_RU', 'gigant/m-ailabs_speech_dataset_ru', 'Vikhrmodels/m-ailabs_ru']
  ок: версия модулей актуальная.

forced-aligner:
  ок: ctc_forced_aligner импортируется (c:\Users\Roman\Documents\Projects\STT_russian_lang\venv\Lib\site-packages\ctc_forced_aligner\__init__.py)

датасеты:
  ок: datasets 2.20.0
  M-AILABS на Hub (поиск): ['gigant/m-ailabs_speech_dataset_fr', 'Attention/mailabs_fyw', 'psiyou/m-ailabs-it_IT', 'tsnngw/mailabs-csv', 'intexcp/M-AILABS-filtred', 'teshtvele/m_ailabs_ru', 'johannhartmann/mailabs-ramona-deininger-raw', 'johannhartmann/mailabs-karlsson-raw']


## 2. Данные: M-AILABS (русские аудиокниги)

###  если HuggingFace недоступен

```bash
# в терминале, из папки проекта, тем же python что у ядра:
python prepare_mailabs.py --limit 4000
# или, если архив скачали вручную браузером:
python prepare_mailabs.py --no-download --tgz ru_RU.tgz --limit 4000
```

In [4]:
# --- ЗАГРУЗКА ИЗ PKL (рекомендуется при недоступном HF) ---
from modules.data import examples_from_pkl
all_examples = examples_from_pkl('mailabs_examples.pkl', cfg.data, use_alignment=True)
# делим на train/val
split = int(len(all_examples) * 0.9)
train_examples, val_examples = all_examples[:split], all_examples[split:]
print(f'train: {len(train_examples)} | val: {len(val_examples)}')

FileNotFoundError: [Errno 2] No such file or directory: 'mailabs_examples.pkl'

In [ ]:
USE_ALIGNMENT = True   # False -> быстрый text-only прогон
MIX_FLEURS    = False  # True -> добавить FLEURS к M-AILABS
LIMIT_TRAIN   = 4000   # None = весь train; аудиокниги крупные, начните умеренно
LIMIT_VAL     = 800

train_examples = build_examples(cfg.data, split="train",      limit=LIMIT_TRAIN,
                                use_alignment=USE_ALIGNMENT, source="mailabs", mix_fleurs=MIX_FLEURS)
val_examples   = build_examples(cfg.data, split="validation", limit=LIMIT_VAL,
                                use_alignment=USE_ALIGNMENT, source="mailabs", mix_fleurs=MIX_FLEURS)

print(f"train: {len(train_examples)} | val: {len(val_examples)}")
ex = train_examples[0]
print("слова:", ex.words[:12])
print("есть акустика:", ex.has_acoustic, "| форма:", ex.acoustic.shape)

# распределение классов пунктуации — убедимся, что ? ! … теперь присутствуют
from collections import Counter
c = Counter(x for e in train_examples for x in e.punct_ids)
print("распределение пунктуации:", {PUNCT_LABELS[k]: c[k] for k in sorted(c)})
par = Counter(x for e in train_examples for x in e.para_ids)
print("абзацы (NO_PARA/PARA):", {PARA_LABELS[k]: par[k] for k in sorted(par)})

[load_mailabs] автопоиск нашёл кандидатов: ['teshtvele/m_ailabs_ru']
[load_mailabs] не удалось загрузить M-AILABS. Проверенные имена: ['teshtvele/m_ailabs_ru', 'psiyou/m-ailabs-ru_RU', 'gigant/m-ailabs_speech_dataset_ru', 'Vikhrmodels/m-ailabs_ru']. Последняя ошибка: Dataset 'Vikhrmodels/m-ailabs_ru' doesn't exist on the Hub or cannot be accessed.. Укажите рабочее имя в cfg.data.mailabs_repos.
[build_examples] корпус недоступен — возвращаю демо-примеры (text-only).
ВНИМАНИЕ: используются ДЕМО-примеры (M-AILABS не загрузился).
Это НЕ реальный корпус — метрики на них не имеют смысла.
Укажите рабочее имя датасета в cfg.data.mailabs_repos и перезапустите.
[load_mailabs] автопоиск нашёл кандидатов: ['teshtvele/m_ailabs_ru']
[load_mailabs] не удалось загрузить M-AILABS. Проверенные имена: ['teshtvele/m_ailabs_ru', 'psiyou/m-ailabs-ru_RU', 'gigant/m-ailabs_speech_dataset_ru', 'Vikhrmodels/m-ailabs_ru']. Последняя ошибка: Dataset 'Vikhrmodels/m-ailabs_ru' doesn't exist on the Hub or cannot be 

In [ ]:
# ПРОВЕРКА: реально ли загрузился M-AILABS (а не demo-fallback на 2 примерах).
# Если train < ~50 примеров — корпус НЕ загрузился, метрики будут бессмысленны.
assert len(train_examples) >= 50, (
    f"M-AILABS не загрузился (train={len(train_examples)}). Это demo-fallback!\n"
    "Укажите рабочее имя датасета в cfg.data.mailabs_repos и перезапустите ячейку выше.\n"
    "Найти имя: huggingface.co/datasets?search=m-ailabs"
)
print(f"OK: загружено {len(train_examples)} train / {len(val_examples)} val примеров.")

### Пересчёт нормализации акустики на train
Если включена акустика, грубые `ACOUSTIC_NORM` лучше заменить реальными mean/std.

In [ ]:
from modules.data import compute_acoustic_stats
import pprint; pprint.pprint(compute_acoustic_stats(train_examples))

## 3. Baseline №1 — BiLSTM

In [ ]:
vocab = WordVocab.build(train_examples, min_freq=1, max_size=50000)
print("словарь:", len(vocab))

collate = partial(baseline_collate, pad_id=vocab.pad_id)
train_loader = DataLoader(BaselineDataset(train_examples, vocab, cfg.train.max_len),
                          batch_size=cfg.train.batch_size, shuffle=True, collate_fn=collate)
val_loader   = DataLoader(BaselineDataset(val_examples, vocab, cfg.train.max_len),
                          batch_size=cfg.train.batch_size, shuffle=False, collate_fn=collate)

lstm = build_model("lstm", vocab_size=len(vocab), pad_id=vocab.pad_id, use_acoustic=True)
print("параметров:", sum(p.numel() for p in lstm.parameters()))

In [ ]:
cfg.train.epochs = 8
cfg.train.lr = 1e-3
lstm = train_model(lstm, train_loader, cfg.train, val_loader=val_loader,
                   eval_fn=evaluate, train_examples=train_examples)  # <-- train_examples для автовесов
lstm_metrics = evaluate(lstm, val_loader, torch.device(DEVICE))
print(pretty_report(lstm_metrics))

## 4. Baseline №2 — Transformer с нуля

In [ ]:
transformer = build_model("transformer", vocab_size=len(vocab), pad_id=vocab.pad_id, use_acoustic=True)
print("параметров:", sum(p.numel() for p in transformer.parameters()))

cfg.train.epochs = 10   # трансформеру с нуля нужно больше эпох/данных
cfg.train.lr = 3e-4
transformer = train_model(transformer, train_loader, cfg.train, val_loader=val_loader,
                          eval_fn=evaluate, train_examples=train_examples)
tr_metrics = evaluate(transformer, val_loader, torch.device(DEVICE))
print(pretty_report(tr_metrics))

## 5. Предобученные — RuBERT-base / rubert-tiny2

In [ ]:
PRESET = "rubert-base"   # или "rubert-tiny2" (легче/быстрее)
model_name = PRETRAINED_PRESETS[PRESET]
print("модель:", model_name)

hf_tok = load_hf_tokenizer(model_name)
ptr_collate = partial(pretrained_collate, pad_id=hf_tok.pad_token_id or 0)
ptr_train_loader = DataLoader(PretrainedDataset(train_examples, hf_tok, cfg.train.max_len),
                              batch_size=cfg.train.batch_size, shuffle=True, collate_fn=ptr_collate)
ptr_val_loader   = DataLoader(PretrainedDataset(val_examples, hf_tok, cfg.train.max_len),
                              batch_size=cfg.train.batch_size, shuffle=False, collate_fn=ptr_collate)

pretrained = build_model("pretrained", model_name=model_name, use_acoustic=True)

In [ ]:
cfg.train.epochs = 10   # RuBERT-голове нужно ~8-12 эпох: первые 2-3 эпохи
                         # F1 может быть 0 (warmup + focal), это НОРМАЛЬНО — не пугайтесь.
pretrained = train_model(pretrained, ptr_train_loader, cfg.train, val_loader=ptr_val_loader,
                         is_pretrained=True, eval_fn=evaluate, train_examples=train_examples)
ptr_metrics = evaluate(pretrained, ptr_val_loader, torch.device(DEVICE))
print(pretty_report(ptr_metrics))

### Сравнение моделей

In [ ]:
import pandas as pd
rows = [{"модель": n,
         "punct F1": round(m.get("punct_f1_macro", 0), 3),
         "para F1":  round(m.get("para_f1_macro", 0), 3),
         "cap F1":   round(m.get("cap_f1_macro", 0), 3)}
        for n, m in [("BiLSTM", lstm_metrics), ("Transformer", tr_metrics), (PRESET, ptr_metrics)]]
pd.DataFrame(rows)

## 6. Инференс

In [ ]:
restorer = PunctuationRestorer(lstm, kind="lstm", vocab=vocab, device=DEVICE)
# предобученная: PunctuationRestorer(pretrained, kind="pretrained", hf_tokenizer=hf_tok, device=DEVICE)

print(restorer.restore("привет как дела я давно тебя не видел"))
print(restorer.restore("что это было невероятно я не ожидал такого поворота событий"))

## 7. Встраивание в SpeechToText-пайплайн
Whisper отдаёт слова + тайм-коды; из них считаются те же акустические признаки, что при обучении (паузы → границы, F0 → `?`/`!`).

In [ ]:
pipeline = STTPunctuationPipeline(restorer)

words = "что это было невероятно я не ожидал такого".split()
t, word_ts = 0.0, []
for w in words:
    word_ts.append({"word": w, "start": round(t,2), "end": round(t+0.3,2)})
    t += 0.3 + (0.7 if w in ("было","невероятно","такого") else 0.05)

print("С паузами:", pipeline({"words": words, "word_timestamps": word_ts, "audio": None}))
print("Текст    :", pipeline({"text": " ".join(words)}))

## 8. Настройка против дисбаланса 
Все рычаги в `cfg.train` (модуль `config.py`):

In [ ]:
cfg.train.loss_type = "focal"      # "ce" — обычный взвешенный CrossEntropy
cfg.train.focal_gamma = 2.0        # больше -> сильнее фокус на редких знаках (попробуйте 3.0)
cfg.train.auto_class_weights = True  # автовеса по частоте в train

## 9. Сохранение

In [ ]:

torch.save(lstm.state_dict(), "lstm_punct.pt"); vocab.save("vocab.json")
torch.save(pretrained.state_dict(), "rubert_punct.pt")